<a href="https://colab.research.google.com/github/Virolero24/ZNI-de-Colombia/blob/main/TransicionEnergetica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
df_proyecto=pd.read_csv('//content/drive/MyDrive/Estado_de_la_prestación_del_servicio_de_energía_en_Zonas_No_Interconectadas_20260226.csv',index_col=0)
df_proyecto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_proyecto['ENERGÍA ACTIVA'] = df_proyecto['ENERGÍA ACTIVA'].astype(str).str.replace(',', '', regex=False)

df_proyecto.info()

In [ ]:
df_proyecto['ENERGÍA ACTIVA'] = pd.to_numeric(df_proyecto['ENERGÍA ACTIVA'], errors='coerce')

df_proyecto.info()

In [ ]:
df_proyecto['POTENCIA MÁXIMA'] = df_proyecto['POTENCIA MÁXIMA'].astype(str).str.replace(',', '', regex=False)


In [ ]:
df_proyecto['POTENCIA MÁXIMA'] = pd.to_numeric(df_proyecto['POTENCIA MÁXIMA'], errors='coerce')

df_proyecto.info()

In [ ]:
df_proyecto['ENERGÍA REACTIVA'] = df_proyecto['ENERGÍA REACTIVA'].astype(str).str.replace(',', '', regex=False)

In [ ]:
df_proyecto['ENERGÍA REACTIVA'] = pd.to_numeric(df_proyecto['ENERGÍA REACTIVA'], errors='coerce')

df_proyecto.info()

In [ ]:
df_aux = pd.read_csv('/content/drive/MyDrive/Estado_de_la_prestación_del_servicio_de_energía_en_Zonas_No_Interconectadas_20260226.csv', usecols=['FECHA DE DEMANDA MÁXIMA'])
df_proyecto['FECHA DE DEMANDA MÁXIMA'] = df_aux['FECHA DE DEMANDA MÁXIMA']

In [ ]:
df_proyecto

In [ ]:
df_proyecto['FECHA DE DEMANDA MÁXIMA'] = pd.to_datetime(df_proyecto['FECHA DE DEMANDA MÁXIMA'], errors='coerce')

print(df_proyecto['FECHA DE DEMANDA MÁXIMA'].head())

In [ ]:
df_proyecto

In [ ]:
# 1. Agrupamos por ID y listamos los nombres únicos de DEPARTAMENTO que tiene cada uno
inconsistencias = df_proyecto.groupby('ID DEPATAMENTO')['DEPARTAMENTO'].unique()

# 2. Filtramos solo los que tienen más de un nombre (ej: "Antioquia" y "Antioquía")
duplicados = inconsistencias[inconsistencias.apply(len) > 1]

print(duplicados)

In [ ]:
# 1. Creamos la columna de ID Departamento extrayendo los 2 primeros dígitos de ID MUNICIPIO
# (Lo convertimos a entero para que coincida con nuestro diccionario)
df_proyecto['ID DEPARTAMENTO'] = (df_proyecto['ID MUNICIPIO'] // 1000).astype(int)

# 2. Ahora aplicamos las correcciones con el ID que acabamos de crear
correcciones = {
    13: 'BOLIVAR',
    18: 'CAQUETA',
    27: 'CHOCO',
    88: 'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA',
    94: 'GUAINIA',
    97: 'VAUPES'
}

df_proyecto['DEPARTAMENTO'] = df_proyecto['ID DEPARTAMENTO'].map(correcciones).fillna(df_proyecto['DEPARTAMENTO'])

# 3. Verificamos que ya no haya duplicados
print(df_proyecto.groupby('ID DEPARTAMENTO')['DEPARTAMENTO'].unique())

In [ ]:
df_proyecto.info()

In [ ]:
df_proyecto.shape

In [ ]:
df_proyecto['FECHA DE DEMANDA MÁXIMA']

In [ ]:

import pandas as pd

# Supongamos que df es tu DataFrame y 'fecha_hora' la columna
df_proyecto['FECHA DE DEMANDA MÁXIMA'] = pd.to_datetime(df_proyecto['FECHA DE DEMANDA MÁXIMA'])

# Crear columna de fecha
#df_proyecto['FECHA DEMANDA MÁXIMA'] = df_proyecto['FECHA DE DEMANDA MÁXIMA'].dt.date

# Crear columna de hora
df_proyecto['HORA DEMANDA MÁXIMA'] = df_proyecto['FECHA DE DEMANDA MÁXIMA'].dt.time

df_proyecto

In [ ]:
#df_proyecto = df_proyecto.drop('HORAS DE DEMANDA MÁXIMA', axis=1)
df_proyecto.info()

In [ ]:
#df_proyecto = df_proyecto.drop('FECHA DE DEMANDA MÁXIMA', axis= 1)
df_proyecto.info()

In [ ]:
print(df_proyecto[['ENERGÍA ACTIVA', 'POTENCIA MÁXIMA', 'PROMEDIO DIARIO EN HORAS']].corr())

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

#Separamos los datos que sí tenemos de los que faltan
df_completo = df_proyecto[df_proyecto['POTENCIA MÁXIMA'].notnull()]
df_faltante = df_proyecto[df_proyecto['POTENCIA MÁXIMA'].isnull()]

#Preparamos el modelo (X = Energía Activa, y = Potencia Máxima)
# Usamos .values.reshape(-1, 1) porque Scikit-Learn espera una matriz
X_train = df_completo[['ENERGÍA ACTIVA']]
y_train = df_completo['POTENCIA MÁXIMA']

modelo = LinearRegression()
modelo.fit(X_train, y_train)

#Predecimos los valores para esos 80 huecos
X_test = df_faltante[['ENERGÍA ACTIVA']]
predicciones = modelo.predict(X_test)

#Llenamos los nulos en el DataFrame original
df_proyecto.loc[df_proyecto['POTENCIA MÁXIMA'].isnull(), 'POTENCIA MÁXIMA'] = predicciones


In [ ]:
print(df_proyecto.isnull().sum())

In [ ]:
df_proyecto['DÍA DE DEMANDA MÁXIMA'] = df_proyecto['FECHA DE DEMANDA MÁXIMA'].dt.day_name()

dias_esp = {
    'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles',
    'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
}
df_proyecto['DÍA DE DEMANDA MÁXIMA'] = df_proyecto['DÍA DE DEMANDA MÁXIMA'].map(dias_esp)


In [ ]:
df_proyecto.info()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Diagnóstico rápido de Calidad (Nulos y Duplicados)
print(f"Nulos:\n{df_proyecto.isnull().sum()}\n\nDuplicados: {df_proyecto.duplicated().sum()}")

# 2. Selección y Normalización (Preparación para la IA)
# Elegimos las variables clave para el análisis de consumo
cols = ['ENERGÍA ACTIVA', 'POTENCIA MÁXIMA', 'PROMEDIO DIARIO EN HORAS']
scaler = StandardScaler()

# Creamos el nuevo DataFrame escalado
df_escalado = pd.DataFrame(scaler.fit_transform(df_proyecto[cols]), columns=cols)

# 3. Resultado final para el informe
print("\n--- Vista previa de datos normalizados ---")
print(df_escalado.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Configuración de estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 5))

# 2. Análisis Univariado: Distribución de la Energía Activa
plt.subplot(1, 3, 1)
sns.histplot(df_proyecto['ENERGÍA ACTIVA'], log_scale=True, color="orange")
plt.title('Distribución de Energía Activa')

# 3. Análisis Bivariado: Relación entre Potencia Máxima y Energía
plt.subplot(1, 3, 2)
sns.scatterplot(data=df_proyecto, x='POTENCIA MÁXIMA', y='ENERGÍA ACTIVA', alpha=0.5)
plt.title('Potencia vs Energía')

# 4. Análisis Multivariado: Mapa de Calor de Correlaciones
plt.subplot(1, 3, 3)
# Solo columnas numéricas para la correlación
columnas_ia = ['ENERGÍA ACTIVA', 'POTENCIA MÁXIMA', 'PROMEDIO DIARIO EN HORAS']
sns.heatmap(df_proyecto[columnas_ia].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlación')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# 1. Definimos el número de grupos (Clusters).
# Probaremos con 3: Bajo, Medio y Alto consumo/potencia.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# 2. Entrenamos el modelo usando los datos NORMALIZADOS
clusters = kmeans.fit_predict(df_escalado)

# 3. Agregamos la categoría resultante a nuestro dataframe original
# para que sea fácil de leer
df_proyecto['CATEGORIA_ZNI'] = clusters

# 4. Ver de qué tamaño quedó cada grupo
print("--- Cantidad de localidades por Categoría ---")
print(df_proyecto['CATEGORIA_ZNI'].value_counts())

# 5. Ver el promedio real de cada grupo para entender qué significan
print("\n--- Perfil promedio por Categoría (Valores Reales) ---")
resumen = df_proyecto.groupby('CATEGORIA_ZNI')[['ENERGÍA ACTIVA', 'POTENCIA MÁXIMA', 'PROMEDIO DIARIO EN HORAS']].mean()
print(resumen)

In [ ]:
# 1. Filtramos el DataFrame original para quedarnos solo con la Categoría 1
df_categoria_1 = df_proyecto[df_proyecto['CATEGORIA_ZNI'] == 1]

# 2. Agrupamos por Municipio y sumamos su Energía Activa
# (En caso de que un municipio tenga varias localidades en esa categoría)
top_municipios = df_categoria_1.groupby('MUNICIPIO')['ENERGÍA ACTIVA'].sum().sort_values(ascending=False)

# 3. Mostramos los 3 primeros
print("--- Top 3 Municipios con Mayor Consumo (Categoría 1) ---")
print(top_municipios.head(5))

# 4. Opcional: Si quieres ver también a qué departamento pertenecen
top_detallado = df_categoria_1.groupby(['DEPARTAMENTO', 'MUNICIPIO'])['ENERGÍA ACTIVA'].sum().sort_values(ascending=False)
print("\n--- Detalle por Departamento ---")
print(top_detallado.head(5))

In [ ]:
# 1. Limpieza de nombres para evitar duplicados por tildes o espacios
df_proyecto['MUNICIPIO'] = df_proyecto['MUNICIPIO'].str.replace('Á', 'A').str.replace('É', 'E').str.replace('Í', 'I').str.replace('Ó', 'O').str.replace('Ú', 'U').str.strip()

# 2. Filtramos nuevamente la Categoría 1 (los 60 registros de alto consumo)
df_cat1 = df_proyecto[df_proyecto['CATEGORIA_ZNI'] == 1]

# 3. Agrupamos por Municipio y sumamos su Energía Activa
top_3_municipios = df_cat1.groupby('MUNICIPIO')['ENERGÍA ACTIVA'].sum().sort_values(ascending=False).head(3)

print("--- Los 3 Municipios de Mayor Consumo en la Categoría 1 ---")
print(top_3_municipios)

# 4. Para tu informe: ¿A qué departamentos pertenecen estos municipios?
print("\n--- Ubicación Geográfica del Top 3 ---")
print(df_cat1[df_cat1['MUNICIPIO'].isin(top_3_municipios.index)].groupby('MUNICIPIO')['DEPARTAMENTO'].unique())

In [ ]:
# 1. Ya sabemos que el #1 es San Andrés.
# Ahora busquemos en la Categoría 2 quiénes son los que más consumen.
df_cat2 = df_proyecto[df_proyecto['CATEGORIA_ZNI'] == 2]

# 2. Agrupamos por Municipio para ver los totales de esa categoría
top_otros = df_cat2.groupby('MUNICIPIO')['ENERGÍA ACTIVA'].sum().sort_values(ascending=False)

print("--- Los otros líderes de consumo (Categoría 2) ---")
print(top_otros.head(2)) # Pedimos los 2 mejores de aquí para completar el Top 3 con San Andrés

In [ ]:
# 1. Agrupamos TODO el dataset por municipio, sin importar la categoría de la IA
top_real = df_proyecto.groupby('MUNICIPIO')['ENERGÍA ACTIVA'].sum().sort_values(ascending=False)

print("--- El Top 3 Real de Consumo de todo el Proyecto ---")
print(top_real.head(3))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_percentage_error

# --- INICIO DEL BLOQUE DE COMPARACIÓN ---
# Usaremos a San Andrés como el "laboratorio" para decidir el mejor modelo
muni_test = 'SAN ANDRES'
df_test = df_proyecto[df_proyecto['MUNICIPIO'] == muni_test].groupby('AÑO SERVICIO')['ENERGÍA ACTIVA'].sum().reset_index()

# Preparamos X (años) y y (consumo)
X_test_all = df_test[['AÑO SERVICIO']]
y_test_all = df_test['ENERGÍA ACTIVA']

# Dividimos para validar: entrenamos con el pasado, probamos con el último año real
X_train_v, y_train_v = X_test_all[:-1], y_test_all[:-1]
X_val, y_val = X_test_all.tail(1), y_test_all.tail(1)

# Definimos modelos con semilla (random_state=42)
mod_lineal = LinearRegression()
mod_arbol = DecisionTreeRegressor(random_state=42)

# Entrenamos ambos
mod_lineal.fit(X_train_v, y_train_v)
mod_arbol.fit(X_train_v, y_train_v)

# Calculamos errores
err_lin = mean_absolute_percentage_error(y_val, mod_lineal.predict(X_val))
err_arb = mean_absolute_percentage_error(y_val, mod_arbol.predict(X_val))

print(f"--- Validación de Modelos (Caso: {muni_test}) ---")
print(f"Error Regresión Lineal: {err_lin:.2%}")
print(f"Error Árbol de Decisión: {err_arb:.2%}")

# Elegimos el mejor para el resto del código
if err_lin <= err_arb:
    modelo_final = LinearRegression()
    print("Resultado: Se seleccionó Regresión Lineal por mayor precisión.")
else:
    modelo_final = DecisionTreeRegressor(random_state=42)
    print("Resultado: Se seleccionó Árbol de Decisión por mayor precisión.")
# --- FIN DEL BLOQUE DE COMPARACIÓN ---

# Ahora puedes seguir con el bucle 'for' que hace las proyecciones a 2028
# usando 'modelo_final' en lugar de definir uno nuevo cada vez.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# 1. Definimos nuestro Top 3 real (ajusta los nombres según tus resultados)
# Asumiendo que fueron San Andrés y los otros dos que te salieron:
top_3_nombres = top_real.head(3).index.tolist()

plt.figure(figsize=(12, 6))

print(f"--- Proyecciones de Consumo para 2028 ---")

for municipio in top_3_nombres:
    # Filtramos datos históricos por municipio
    datos_muni = df_proyecto[df_proyecto['MUNICIPIO'] == municipio].groupby('AÑO SERVICIO')['ENERGÍA ACTIVA'].sum().reset_index()

    # Preparar modelo
    X = datos_muni[['AÑO SERVICIO']]
    y = datos_muni['ENERGÍA ACTIVA']

    if len(X) > 1: # Solo si hay más de un año de datos para trazar una línea
        modelo = LinearRegression().fit(X, y)

        # Predecir hasta 2028
        años_proyeccion = pd.DataFrame({'AÑO SERVICIO': [2025, 2026, 2027, 2028]})
        predicciones = modelo.predict(años_proyeccion)

        # Mostrar resultado por consola
        print(f"{municipio}: Estima {predicciones[-1]:,.2f} kWh para 2028")

        # Graficar
        linea = plt.plot(datos_muni['AÑO SERVICIO'], y, marker='o', label=f'Histórico {municipio}')
        plt.plot(range(2025, 2029), predicciones, linestyle='--', color=linea[0].get_color(), alpha=0.7)

plt.title('Tendencia y Proyección de Consumo Energético a 2028 (Top 3 Municipios ZNI)')
plt.xlabel('Año')
plt.ylabel('Energía Activa (kWh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Calcular el consumo total de toda la base de datos
consumo_total_zni = df_proyecto['ENERGÍA ACTIVA'].sum()

# Calcular cuánto suman tus 3 municipios proyectados
consumo_top3 = df_proyecto[df_proyecto['MUNICIPIO'].isin(['SAN ANDRES', 'LETICIA', 'PUERTO CARREÑO'])]['ENERGÍA ACTIVA'].sum()

porcentaje_impacto = (consumo_top3 / consumo_total_zni) * 100

print(f"El Top 3 representa el {porcentaje_impacto:.2f}% del consumo total de las ZNI.")

In [ ]:
from google.colab import files

# Suponiendo que tu DataFrame final se llama 'df'
df_proyecto.to_csv('datos_finales_zni.csv', index=False)